[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/08_knn/90_exercises.ipynb)

# Bài tập — Chương 08: K-Nearest Neighbors (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/08_knn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import load_digits_data, make_classification_2d
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết 3 bước dự đoán của KNN (phân loại và hồi quy). Vì sao KNN được gọi là *lazy* và *non-parametric*? Viết công thức khoảng cách Minkowski.

✍️ Trả lời:

**Đáp án:** (1) tính khoảng cách tới mọi điểm train; (2) chọn $k$ điểm gần nhất; (3) bỏ phiếu đa số (phân loại) / lấy trung bình (hồi quy). *Lazy*: không học gì lúc train, chỉ ghi nhớ dữ liệu; mọi tính toán dồn vào lúc dự đoán. *Non-parametric*: "model" chính là toàn bộ dữ liệu train, độ phức tạp tăng theo $n$. Minkowski: $d_p(\mathbf a, \mathbf b) = \left(\sum_j |a_j - b_j|^p\right)^{1/p}$.

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích cho một người không học ML: vì sao KNN gần như **bắt buộc** phải scaling feature, trong khi Decision Tree thì không?

✍️ Trả lời:

**Gợi ý:** KNN đo "giống nhau" bằng khoảng cách — cộng gộp chênh lệch của mọi feature. Feature có đơn vị lớn (thu nhập theo đồng) sẽ áp đảo feature đơn vị nhỏ (tuổi theo năm), nên "láng giềng" chỉ còn là "người có thu nhập gần giống". Đổi đơn vị là đổi luôn kết quả. Decision Tree chỉ so sánh **từng feature với một ngưỡng** ($x_j \le t$), thứ tự giá trị không đổi khi scale → kết quả không đổi.

## L2 — Tính tay

**Câu 3.** Dữ liệu train 1D: $x = [1, 2, 4, 7, 8]$, nhãn $y = [0, 0, 1, 1, 1]$, giá trị hồi quy $t = [10, 12, 20, 30, 34]$. Điểm truy vấn $x_q = 3.2$.
- (a) Dự đoán lớp với $k = 3$ (uniform).
- (b) Dự đoán hồi quy với $k = 2$ (uniform).
- (c) Dự đoán hồi quy với $k = 2$, trọng số $w = 1/d$.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
pred_class_k3 = 0
pred_reg_k2 = (12 + 20) / 2
pred_reg_k2_weighted = (12 / 1.2 + 20 / 0.8) / (1 / 1.2 + 1 / 0.8)
### END CODE HERE ###

In [ ]:
check_equal(pred_class_k3, 0, "k=3: láng giềng x=2 (0), x=4 (1), x=1 (0) → lớp 0")
check_close(pred_reg_k2, 16.0, message="hồi quy k=2")
check_close(pred_reg_k2_weighted, 16.8, tol=1e-6, message="hồi quy k=2, trọng số 1/d")

**Câu 4.** Hai điểm $\mathbf a = (1, 2, 3)$, $\mathbf b = (4, 0, 3)$. Tính khoảng cách Euclid, Manhattan, Chebyshev ($p = \infty$) và cosine distance.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
d_euclid = np.sqrt(9 + 4 + 0)
d_manhattan = 3 + 2 + 0
d_chebyshev = 3
d_cosine = 1 - (4 + 0 + 9) / (np.sqrt(14) * 5)
### END CODE HERE ###

In [ ]:
check_close(d_euclid, np.sqrt(13), message="Euclid")
check_close(d_manhattan, 5, message="Manhattan")
check_close(d_chebyshev, 3, message="Chebyshev")
check_close(d_cosine, 1 - 13 / (5 * np.sqrt(14)), message="cosine distance")

## L3 — Code

**Bài 5.** Viết `euclidean_matrix(A, B)` — ma trận khoảng cách Euclid $(n_A \times n_B)$ **không dùng vòng lặp**, theo công thức $\lVert\mathbf a - \mathbf b\rVert^2 = \lVert\mathbf a\rVert^2 + \lVert\mathbf b\rVert^2 - 2\mathbf a^\top\mathbf b$ (nhớ chặn giá trị âm do sai số làm tròn).

In [ ]:
def euclidean_matrix(A, B):
    ### START CODE HERE ### (≈ 2 dòng)
    sq = (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T
    D = np.sqrt(np.maximum(sq, 0))
    ### END CODE HERE ###
    return D

In [ ]:
from scipy.spatial.distance import cdist

A_, B_ = rng.normal(size=(30, 5)), rng.normal(size=(40, 5))
check_close(euclidean_matrix(A_, B_), cdist(A_, B_), tol=1e-8, message="khớp scipy cdist")
check_true(np.all(np.diag(euclidean_matrix(A_, A_)) >= 0), "không có NaN/giá trị âm trên đường chéo")

**Bài 6.** Viết `knn_classify(X_train, y_train, X_test, k)` dùng `euclidean_matrix` + `np.argpartition` + bỏ phiếu (`np.bincount`). Nhãn là số nguyên $0..C-1$. Đạt accuracy giống `KNeighborsClassifier` trên digits.

In [ ]:
def knn_classify(X_train, y_train, X_test, k=5):
    ### START CODE HERE ### (≈ 4 dòng)
    D = euclidean_matrix(X_test, X_train)
    idx = np.argpartition(D, k - 1, axis=1)[:, :k]
    n_classes = y_train.max() + 1
    pred = np.array([np.bincount(y_train[row], minlength=n_classes).argmax() for row in idx])
    ### END CODE HERE ###
    return pred

In [ ]:
Xd, yd, _ = load_digits_data()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=0, stratify=yd)
acc_ours = np.mean(knn_classify(Xd_tr, yd_tr, Xd_te, k=5) == yd_te)
acc_sk = KNeighborsClassifier(5).fit(Xd_tr, yd_tr).score(Xd_te, yd_te)
print(f"scratch = {acc_ours:.4f} | sklearn = {acc_sk:.4f}")
check_close(acc_ours, acc_sk, tol=0.005, message="accuracy ≈ sklearn")

## L4 — Debug

**Bài 7.** Pipeline dưới đây có **2 lỗi**: (1) scaler được fit trên **toàn bộ** dữ liệu trước khi chia (leakage); (2) chọn $k$ bằng accuracy **trên tập train** → luôn chọn $k = 1$. Sửa: dùng `Pipeline` + chọn $k$ bằng 5-fold CV **chỉ trên tập train**, lưu vào `best_k`, rồi đánh giá trên test (`test_acc`).

In [ ]:
Xb, yb = make_classification_2d("moons", n=800, noise=0.35, seed=11)
Xb = Xb * [1, 50]                       # feature 2 có thang đo lớn hơn
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=0, stratify=yb)
### START CODE HERE ### (≈ 5 dòng)
# STARTER: X_all = StandardScaler().fit_transform(Xb)                    # 🐛 leakage
# STARTER: X_tr_s, X_te_s = X_all[: len(Xb_tr)], X_all[len(Xb_tr):]      # 🐛 còn sai thứ tự hàng!
# STARTER: scores = {k: KNeighborsClassifier(k).fit(X_tr_s, yb_tr).score(X_tr_s, yb_tr) for k in range(1, 60, 2)}  # 🐛 train acc
# STARTER: best_k = max(scores, key=scores.get)
# STARTER: test_acc = KNeighborsClassifier(best_k).fit(X_tr_s, yb_tr).score(X_te_s, yb_te)
scores = {k: cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), Xb_tr, yb_tr, cv=5).mean()
          for k in range(1, 60, 2)}
best_k = max(scores, key=scores.get)
final_model = make_pipeline(StandardScaler(), KNeighborsClassifier(best_k)).fit(Xb_tr, yb_tr)
test_acc = final_model.score(Xb_te, yb_te)
### END CODE HERE ###
print(f"best_k = {best_k} | test accuracy = {test_acc:.4f}")

In [ ]:
check_greater(best_k, 1, "k được chọn bằng CV, không phải k = 1")
check_greater(test_acc, 0.85, "test accuracy > 0.85")

## L5 — Phân tích

**Bài 8.** Trên dữ liệu `moons` (`n=1000`, `noise=0.3`), thêm lần lượt 0, 2, 8, 32 feature **nhiễu** Gaussian. Với mỗi trường hợp, đo accuracy (5-fold CV) của KNN ($k = 15$, có scaling) và Logistic Regression với feature mapping bậc 3 — nhưng **chỉ trên 2 feature đầu** cho Logistic (giả sử biết feature nào có ích) và trên **tất cả** feature cho KNN. Lưu vào dict `acc_knn[n_noise]`. Vẽ đường accuracy theo số feature nhiễu và nhận xét.

In [ ]:
Xm, ym = make_classification_2d("moons", n=1000, noise=0.3, seed=5)
acc_knn = {}
for n_noise in [0, 2, 8, 32]:
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: X_noisy = None      # ghép Xm với n_noise cột nhiễu N(0, 1)
    # STARTER: model = None        # Pipeline: StandardScaler + KNN(15)
    # STARTER: acc_knn[n_noise] = None
    X_noisy = np.hstack([Xm, rng.normal(size=(len(Xm), n_noise))])
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(15))
    acc_knn[n_noise] = cross_val_score(model, X_noisy, ym, cv=5).mean()
    ### END CODE HERE ###
print({k: round(v, 3) for k, v in acc_knn.items()})
plt.plot(list(acc_knn), list(acc_knn.values()), "o-")
plt.xlabel("số feature nhiễu")
plt.ylabel("accuracy (5-fold CV)")
plt.title("KNN suy giảm khi thêm feature vô ích")
plt.show()

In [ ]:
check_greater(acc_knn[0] - acc_knn[32], 0.05, "accuracy giảm rõ khi thêm 32 feature nhiễu")

✍️ Nhận xét:

**Gợi ý:** mỗi feature nhiễu (đã scale về phương sai 1) đóng góp vào khoảng cách đúng bằng một feature có ích → với 32 feature nhiễu, 94% "khoảng cách" là nhiễu, láng giềng gần như ngẫu nhiên so với cấu trúc thật. Đây là curse of dimensionality + feature vô ích. Khắc phục: feature selection, PCA, học metric, hoặc dùng model tự chọn feature (tree-based).

## L6 — Thực tế

**Bài 9.** Xây model KNN dự đoán **giá nhà** (`house_prices.csv`, target `price_billion_vnd`): chọn các cột số hợp lý, log-transform diện tích, scaling, chọn `n_neighbors` và `weights` bằng `GridSearchCV` (5-fold, MAE). Lưu MAE trên tập test vào `test_mae` và so sánh với baseline "dự đoán trung vị" (`baseline_mae`). Mục tiêu: `test_mae < 0.5 * baseline_mae`.

In [ ]:
houses = pd.read_csv(repo_path("data", "sample", "house_prices.csv"))
print(houses.columns.tolist())
### START CODE HERE ### (≈ 10 dòng)
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV

num_cols = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
Xh = houses[num_cols].copy()
Xh["area_m2"] = np.log(Xh["area_m2"])
yh = houses["price_billion_vnd"]
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.25, random_state=0)
grid = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsRegressor()),
                    {"kneighborsregressor__n_neighbors": [3, 5, 10, 20, 40], "kneighborsregressor__weights": ["uniform", "distance"]},
                    cv=5, scoring="neg_mean_absolute_error").fit(Xh_tr, yh_tr)
test_mae = mean_absolute_error(yh_te, grid.predict(Xh_te))
baseline_mae = mean_absolute_error(yh_te, np.full(len(yh_te), yh_tr.median()))
### END CODE HERE ###
print(grid.best_params_)
print(f"MAE test = {test_mae:.3f} tỉ | baseline trung vị = {baseline_mae:.3f} tỉ")

In [ ]:
check_less(test_mae, 0.5 * baseline_mae, "MAE < 50% baseline")

## L7 — Nâng cao

**Bài 10.** **Condensed Nearest Neighbor (Hart, 1968)** — giảm số điểm phải lưu. Thuật toán: bắt đầu với tập $S$ gồm 1 điểm mỗi lớp; lặp qua dữ liệu train, nếu 1-NN dùng $S$ phân loại **sai** điểm $x_i$ thì thêm $x_i$ vào $S$; lặp lại tới khi một lượt không thêm điểm nào. Cài đặt `condensed_nn(X, y)` trả về **chỉ số** các điểm giữ lại. Trên digits: giữ ≤ 15% dữ liệu train mà 1-NN accuracy test giảm < 5 điểm phần trăm. Nhận xét: đánh đổi bộ nhớ/tốc độ lấy bao nhiêu accuracy?

In [ ]:
def condensed_nn(X, y, seed=0):
    r = np.random.default_rng(seed)
    order = r.permutation(len(X))
    keep = [order[np.argmax(y[order] == c)] for c in np.unique(y)]     # 1 điểm mỗi lớp
    changed = True
    while changed:
        changed = False
        for i in order:
            ### START CODE HERE ### (≈ 4 dòng)
            # STARTER: nearest = None   # chỉ số (trong X) của điểm thuộc keep gần X[i] nhất
            # STARTER: # nếu nhãn của nearest khác y[i]: thêm i vào keep và đặt changed = True
            nearest = keep[int(np.argmin(np.linalg.norm(X[keep] - X[i], axis=1)))]
            if y[nearest] != y[i]:
                keep.append(i)
                changed = True
            ### END CODE HERE ###
    return np.array(keep)

In [ ]:
kept = condensed_nn(Xd_tr, yd_tr)
frac = len(kept) / len(Xd_tr)
acc_full = KNeighborsClassifier(1).fit(Xd_tr, yd_tr).score(Xd_te, yd_te)
acc_cnn = KNeighborsClassifier(1).fit(Xd_tr[kept], yd_tr[kept]).score(Xd_te, yd_te)
print(f"giữ {len(kept)}/{len(Xd_tr)} điểm ({frac:.1%}) | 1-NN đầy đủ = {acc_full:.4f} | condensed = {acc_cnn:.4f}")
check_less(frac, 0.15, "giữ ≤ 15% dữ liệu")
check_less(acc_full - acc_cnn, 0.05, "accuracy giảm < 5 điểm %")

✍️ Nhận xét:

**Gợi ý:** chỉ giữ ~10% điểm (những điểm gần ranh giới giữa các lớp — điểm "nội địa" của một lớp là thừa) → dự đoán nhanh và nhẹ hơn ~10 lần, accuracy giảm vài điểm phần trăm vì CNN nhạy với nhiễu (điểm nhiễu luôn bị phân loại sai nên luôn được giữ) và phụ thuộc thứ tự duyệt. Cải tiến: lọc nhiễu trước (Edited NN), rồi mới condense.